In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData"
)

RAW_DIR = DRIVE_ROOT / "raw"
PROCESSED_DIR = DRIVE_ROOT / "processed"
MANIFEST_DIR = DRIVE_ROOT / "manifests"
OUTPUT_DIR = DRIVE_ROOT / "outputs" / "cs1_exp0_lr"

for directory in [RAW_DIR, PROCESSED_DIR, MANIFEST_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Drive data root:", DRIVE_ROOT)
print("Raw data directory:", RAW_DIR)
print("EXP-0 output directory:", OUTPUT_DIR)

Drive data root: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData
Raw data directory: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/raw
EXP-0 output directory: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_exp0_lr


In [3]:
from pathlib import Path
import os
import shutil
import sys

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
BRANCH = "prashant"
REPO_DIR = Path("/content/DiverseVul--IS-Project")

if not REPO_DIR.exists():
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    os.chdir(REPO_DIR)
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}

PROJECT_DIR = REPO_DIR / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.chdir(PROJECT_DIR)

print("Repository:", REPO_DIR)
print("Project directory:", PROJECT_DIR)
print("Current working directory:", Path.cwd())

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
Unpacking objects: 100% (6/6), 1.24 KiB | 212.00 KiB/s, done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
From https://github.com/EnomisLP/DiverseVul--IS-Project
 * branch            prashant   -> FETCH_HEAD
   3876f35..1015a16  prashant   -> origin/prashant
Already on 'prashant'
Your branch is behind 'origin/prashant' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/EnomisLP/DiverseVul--IS-Project
 * branch            prashant   -> FETCH_HEAD
Updating 3876f35..1015a16
Fast-forward
 vuln-detection/src/case_study_1/exp0_lr.py | 86 +++++++++++++++++-------------
 1 file changed, 50 insertions(+), 36 deletions(-)
Repository: /content/DiverseVul--IS-Project
Project directory: /content/DiverseVul--IS-Project/vuln-detection
Current working directory: /content/DiverseVul--IS-P

In [4]:
!pip -q install \
    numpy \
    pandas \
    scipy \
    scikit-learn \
    matplotlib \
    pyyaml \
    pyarrow \
    joblib

In [5]:
from pathlib import Path

print("Files found in RAW_DIR:")
for file_path in sorted(RAW_DIR.iterdir()):
    print(" -", file_path.name)

Files found in RAW_DIR:
 - rdiversevul.json


In [6]:
DATASET_PATH = RAW_DIR / "rdiversevul.json"

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATASET_PATH}\n"
        "Check the filename printed above and update DATASET_PATH."
    )

print("Using dataset:", DATASET_PATH)

Using dataset: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/raw/rdiversevul.json


In [7]:
DATASET_PATH = RAW_DIR / "rdiversevul.json"

In [8]:
from case_study_1.dataset_loader import (
    format_audit_report,
    load_and_audit,
)

AUDIT_OUTPUT_PATH = OUTPUT_DIR / "dataset_audit.json"

dataset_df, audit_report = load_and_audit(
    path=DATASET_PATH,
    audit_output_path=AUDIT_OUTPUT_PATH,
    require_project=True,
)

print(format_audit_report(audit_report))

print("\nAudit report saved to:")
print(AUDIT_OUTPUT_PATH)

CASE STUDY 1 — DATASET AUDIT
Rows loaded:                 261,667
Fully usable grouped-CV rows: 261,667
Vulnerable / non-vulnerable: 13,938 / 247,729
Vulnerable rate:             0.053266
Unique projects:             797
Exact duplicate excess rows: 0
Conflicting duplicate hashes: 0
Code length (characters):    median=475.0, q75=1085.0, max=240968
Warnings:
  - Positive class is rare; accuracy must not be a primary metric.

Audit report saved to:
/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_exp0_lr/dataset_audit.json


In [9]:
import case_study_1.normalization as normalization

print("Normalization version:", normalization.NORMALIZATION_VERSION)

Normalization version: cs1-conservative-v1


In [10]:
raw_example = (
    "\ufeffint\tcopy  (char * dst, const char * src) {\r\n"
    "    // keep   this comment\r\n"
    '    const char * note = "two   spaces";   \r\n'
    "    return 0;\x00\r\n"
    "}   \r\n"
)

normalized_example = normalization.normalize_code(raw_example)

print("RAW:")
print(repr(raw_example))

print("\nNORMALIZED:")
print(repr(normalized_example))

print("\nREADABLE NORMALIZED CODE:")
print(normalized_example)

assert "\r" not in normalized_example
assert "\x00" not in normalized_example
assert '// keep   this comment' in normalized_example
assert '"two   spaces"' in normalized_example
assert normalization.normalize_code(normalized_example) == normalized_example

print("\nNormalization smoke test passed.")

RAW:
'\ufeffint\tcopy  (char * dst, const char * src) {\r\n    // keep   this comment\r\n    const char * note = "two   spaces";   \r\n    return 0;\x00\r\n}   \r\n'

NORMALIZED:
'int copy (char * dst, const char * src) {\n// keep   this comment\nconst char * note = "two   spaces";\nreturn 0;\n}'

READABLE NORMALIZED CODE:
int copy (char * dst, const char * src) {
// keep   this comment
const char * note = "two   spaces";
return 0;
}

Normalization smoke test passed.


In [11]:
from pathlib import Path
import json

NORMALIZED_DATA_PATH = (
    PROCESSED_DIR / "rdiversevul_cs1_normalized_v1.parquet"
)

NORMALIZATION_REPORT_PATH = (
    OUTPUT_DIR / "normalization_summary.json"
)

dataset_df = normalization.add_normalized_code_column(
    frame=dataset_df,
    source_column="code",
    target_column="normalized_code",
)

normalization_report = normalization.normalization_summary(
    raw_codes=dataset_df["code"],
    normalized_codes=dataset_df["normalized_code"],
)

print(json.dumps(normalization_report, indent=2))

{
  "normalization_version": "cs1-conservative-v1",
  "rows": 261667,
  "rows_changed": 260243,
  "rows_changed_rate": 0.9945579687159635,
  "raw_empty_rows": 0,
  "normalized_empty_rows": 0,
  "raw_characters_total": 287057543,
  "normalized_characters_total": 246675768,
  "median_raw_characters": 475.0,
  "median_normalized_characters": 425.0
}


In [12]:
from IPython.display import display

changed_rows = dataset_df.loc[
    dataset_df["code"] != dataset_df["normalized_code"],
    ["source_row_id", "label", "project", "code", "normalized_code"],
].sample(
    n=min(5, int((dataset_df["code"] != dataset_df["normalized_code"]).sum())),
    random_state=42,
)

display(changed_rows)

,source_row_id,label,project,code,normalized_code
192491,192491,0,evolution-data-server,e_data_server_util_get_dbus_call_timeout (void...,e_data_server_util_get_dbus_call_timeout (void...
57329,57329,0,linux,void intel_uc_fw_change_status(struct intel_uc...,void intel_uc_fw_change_status(struct intel_uc...
45821,45821,0,cpython,"Subscript(expr_ty value, slice_ty slice, expr_...","Subscript(expr_ty value, slice_ty slice, expr_..."
145770,145770,0,postgres,"internal_putbytes(const char *s, size_t len) {...","internal_putbytes(const char *s, size_t len)\n..."
101738,101738,0,envoy,"TEST_F(GroupVerifierTest, TestRequiresAnyLastI...","TEST_F(GroupVerifierTest, TestRequiresAnyLastI..."


In [13]:
assert len(dataset_df) == 261_667, "Row count changed unexpectedly."
assert dataset_df["normalized_code"].notna().all(), "Missing normalized code found."
assert (dataset_df["normalized_code"].str.strip() != "").all(), (
    "Normalization created empty code samples."
)

assert (
    dataset_df["code"].str.len().median()
    >= dataset_df["normalized_code"].str.len().median()
), "Unexpected length increase after normalization."

print("Normalization completed successfully.")
print(f"Rows retained: {len(dataset_df):,}")
print(
    "Changed rows: "
    f"{normalization_report['rows_changed']:,} "
    f"({normalization_report['rows_changed_rate']:.2%})"
)
print(
    "Median raw length: "
    f"{normalization_report['median_raw_characters']:.1f} characters"
)
print(
    "Median normalized length: "
    f"{normalization_report['median_normalized_characters']:.1f} characters"
)

Normalization completed successfully.
Rows retained: 261,667
Changed rows: 260,243 (99.46%)
Median raw length: 475.0 characters
Median normalized length: 425.0 characters


In [14]:
import re
import unicodedata

def remove_formatting_only(code):
    """
    Build a simplified fingerprint for checking whether normalization removed
    or changed non-whitespace characters.

    This intentionally ignores whitespace differences but keeps all remaining
    tokens, identifiers, operators, punctuation, literals, and API names.
    """
    text = str(code)

    text = unicodedata.normalize("NFKC", text)
    text = text.lstrip("\ufeff")
    text = text.replace("\x00", "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    return re.sub(r"\s+", "", text)


# Check a deterministic random sample first.
CHECK_SAMPLE_SIZE = min(10_000, len(dataset_df))

check_df = dataset_df.sample(
    n=CHECK_SAMPLE_SIZE,
    random_state=42,
).copy()

check_df["raw_fingerprint"] = check_df["code"].map(remove_formatting_only)
check_df["normalized_fingerprint"] = check_df["normalized_code"].map(
    remove_formatting_only
)

mismatches = check_df.loc[
    check_df["raw_fingerprint"] != check_df["normalized_fingerprint"],
    [
        "source_row_id",
        "label",
        "project",
        "code",
        "normalized_code",
        "raw_fingerprint",
        "normalized_fingerprint",
    ],
]

print(f"Rows checked: {len(check_df):,}")
print(f"Non-whitespace mismatches: {len(mismatches):,}")

if len(mismatches) == 0:
    print(" Token-preservation audit passed.")
    print(" No non-whitespace symbols changed in the sampled functions.")
else:
    print(" Mismatches found. Inspect them before proceeding.")
    display(mismatches.head(5))

Rows checked: 10,000
Non-whitespace mismatches: 0
 Token-preservation audit passed.
 No non-whitespace symbols changed in the sampled functions.


In [15]:
from IPython.display import display

changed_df = dataset_df.loc[
    dataset_df["code"] != dataset_df["normalized_code"],
    ["source_row_id", "label", "project", "code", "normalized_code"],
]

sample_examples = changed_df.sample(
    n=min(5, len(changed_df)),
    random_state=42,
)

for _, row in sample_examples.iterrows():
    print("=" * 100)
    print(
        f"source_row_id={row['source_row_id']} | "
        f"label={row['label']} | "
        f"project={row['project']}"
    )

    print("\nRAW CODE:")
    print(row["code"])

    print("\nNORMALIZED CODE:")
    print(row["normalized_code"])

    print("=" * 100)

source_row_id=192491 | label=0 | project=evolution-data-server

RAW CODE:
e_data_server_util_get_dbus_call_timeout (void)
{
	return default_dbus_timeout;
}

NORMALIZED CODE:
e_data_server_util_get_dbus_call_timeout (void)
{
return default_dbus_timeout;
}
source_row_id=57329 | label=0 | project=linux

RAW CODE:
void intel_uc_fw_change_status(struct intel_uc_fw *uc_fw,
			       enum intel_uc_fw_status status)
{
	uc_fw->__status =  status;
	drm_dbg(&__uc_fw_to_gt(uc_fw)->i915->drm,
		"%s firmware -> %s\n",
		intel_uc_fw_type_repr(uc_fw->type),
		status == INTEL_UC_FIRMWARE_SELECTED ?
		uc_fw->path : intel_uc_fw_status_repr(status));
}

NORMALIZED CODE:
void intel_uc_fw_change_status(struct intel_uc_fw *uc_fw,
enum intel_uc_fw_status status)
{
uc_fw->__status = status;
drm_dbg(&__uc_fw_to_gt(uc_fw)->i915->drm,
"%s firmware -> %s\n",
intel_uc_fw_type_repr(uc_fw->type),
status == INTEL_UC_FIRMWARE_SELECTED ?
uc_fw->path : intel_uc_fw_status_repr(status));
}
source_row_id=45821 | label=0 | p

In [16]:
cache_columns = [
    "source_row_id",
    "code",
    "normalized_code",
    "label",
    "project",
]

normalized_cache_df = dataset_df[cache_columns].copy()

normalized_cache_df.to_parquet(
    NORMALIZED_DATA_PATH,
    index=False,
)

with open(NORMALIZATION_REPORT_PATH, "w", encoding="utf-8") as file:
    json.dump(normalization_report, file, indent=2)

print("✅ Normalized Case Study 1 dataset saved.")
print(NORMALIZED_DATA_PATH)

✅ Normalized Case Study 1 dataset saved.
/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_v1.parquet


In [17]:
# from IPython.display import display

# print("Dataset shape:", dataset_df.shape)

# print("\nCanonical columns used by Case Study 1:")
# print(["code", "label", "project", "cwe", "source_row_id"])

# display(
#     dataset_df[
#         ["source_row_id", "code", "label", "project", "cwe"]
#     ].head(5)
# )

In [18]:
# usable_rows = audit_report["rows"]["fully_usable_for_grouped_cv"]
# vulnerable_rows = audit_report["labels"]["vulnerable_1"]
# safe_rows = audit_report["labels"]["non_vulnerable_0"]
# unique_projects = audit_report["projects"]["unique_valid_projects"]

# assert usable_rows > 0, "No rows are usable for project-aware cross-validation."
# assert vulnerable_rows > 0, "No vulnerable functions were detected."
# assert safe_rows > 0, "No non-vulnerable functions were detected."
# assert unique_projects >= 5, (
#     "At least five distinct projects are needed for 5-fold grouped CV."
# )

# print("Dataset loading and initial audit completed successfully.")
# print(f"Usable rows for grouped CV: {usable_rows:,}")
# print(f"Vulnerable rows: {vulnerable_rows:,}")
# print(f"Non-vulnerable rows: {safe_rows:,}")
# print(f"Unique projects: {unique_projects:,}")

In [19]:
import pandas as pd
from pathlib import Path

NORMALIZED_DATA_PATH = (
    PROCESSED_DIR / "rdiversevul_cs1_normalized_v1.parquet"
)

if not NORMALIZED_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Normalized dataset cache was not found:\n{NORMALIZED_DATA_PATH}\n"
        "Run and save the normalization stage first."
    )

normalized_df = pd.read_parquet(NORMALIZED_DATA_PATH)

required_columns = {
    "source_row_id",
    "code",
    "normalized_code",
    "label",
    "project",
}

missing_columns = required_columns - set(normalized_df.columns)
if missing_columns:
    raise ValueError(
        f"Normalized cache is missing required columns: {sorted(missing_columns)}"
    )

print("Normalized dataset shape:", normalized_df.shape)
print("Columns:", normalized_df.columns.tolist())
print("Projects:", normalized_df["project"].nunique())
print("Positive rate:", f"{normalized_df['label'].mean():.2%}")

Normalized dataset shape: (261667, 5)
Columns: ['source_row_id', 'code', 'normalized_code', 'label', 'project']
Projects: 797
Positive rate: 5.33%


In [20]:
import case_study_1.split_manifest as split_manifest

print("Manifest version:", split_manifest.MANIFEST_VERSION)

Manifest version: cs1-project-grouped-5fold-v1


In [21]:
from pathlib import Path

MANIFEST_OUTPUT_DIR = MANIFEST_DIR
MANIFEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

split_config = split_manifest.SplitConfig(
    n_splits=5,
    random_state=42,
    shuffle=True,
)

dataset_fingerprint = split_manifest.dataset_split_fingerprint(
    normalized_df,
    config=split_config,
)

manifest_df = split_manifest.create_project_grouped_manifest(
    normalized_df,
    config=split_config,
)

manifest_paths = split_manifest.save_manifest_artifacts(
    manifest=manifest_df,
    output_dir=MANIFEST_OUTPUT_DIR,
    config=split_config,
    dataset_fingerprint=dataset_fingerprint,
    normalized_dataset_path=NORMALIZED_DATA_PATH,
)

print("Manifest creation completed.")
print("\nSaved artifacts:")
print("CSV:     ", manifest_paths.csv_path)
print("Parquet: ", manifest_paths.parquet_path)
print("Summary: ", manifest_paths.summary_path)
print("Metadata:", manifest_paths.metadata_path)

Manifest creation completed.

Saved artifacts:
CSV:      /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_grouped_5fold_manifest.csv
Parquet:  /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_grouped_5fold_manifest.parquet
Summary:  /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_grouped_5fold_fold_summary.csv
Metadata: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_grouped_5fold_metadata.json


In [22]:
from IPython.display import display

fold_summary = split_manifest.summarize_manifest(
    manifest_df,
    config=split_config,
)

display(
    fold_summary.style.format(
        {
            "test_positive_rate": "{:.4%}",
            "positive_rate_delta_from_global": "{:+.4%}",
            "test_row_share": "{:.2%}",
            "test_project_share": "{:.2%}",
        }
    )
)

,fold,test_rows,test_vulnerable,test_non_vulnerable,test_positive_rate,positive_rate_delta_from_global,test_unique_projects,train_rows,train_unique_projects,train_test_project_overlap,test_row_share,test_project_share
0,0,69256,3397,65859,4.9050%,-0.4216%,54,192411,743,0,26.47%,6.78%
1,1,51771,2798,48973,5.4046%,+0.0780%,188,209896,609,0,19.79%,23.59%
2,2,46043,2325,43718,5.0496%,-0.2770%,192,215624,605,0,17.60%,24.09%
3,3,47064,2667,44397,5.6668%,+0.3401%,181,214603,616,0,17.99%,22.71%
4,4,47533,2751,44782,5.7876%,+0.4609%,182,214134,615,0,18.17%,22.84%


In [23]:
split_manifest.assert_manifest_integrity(
    manifest_df,
    config=split_config,
)

assert len(manifest_df) == len(normalized_df)
assert manifest_df["source_row_id"].nunique() == len(normalized_df)
assert manifest_df["fold"].nunique() == 5
assert manifest_df["fold"].between(0, 4).all()

assert (fold_summary["train_test_project_overlap"] == 0).all()
assert (fold_summary["test_vulnerable"] > 0).all()
assert (fold_summary["test_non_vulnerable"] > 0).all()

print("Every sample has exactly one test-fold assignment.")
print("No project overlaps between train and test within any fold.")
print("Every test fold contains vulnerable and non-vulnerable functions.")

Every sample has exactly one test-fold assignment.
No project overlaps between train and test within any fold.
Every test fold contains vulnerable and non-vulnerable functions.


In [24]:
dataset_with_folds = split_manifest.apply_manifest(
    frame=normalized_df,
    manifest=manifest_df,
)

print("Dataset with fold assignments:", dataset_with_folds.shape)

display(
    dataset_with_folds[
        [
            "source_row_id",
            "label",
            "project",
            "fold",
            "normalized_code",
        ]
    ].head(10)
)

Dataset with fold assignments: (261667, 6)


,source_row_id,label,project,fold,normalized_code
0,0,0,linux,0,"void dcn20_calculate_wm(\nstruct dc *dc, struc..."
1,1,0,net,1,int hw_atl_utils_soft_reset(struct aq_hw_s *se...
2,2,0,net,1,static u32 hw_atl_utils_rpc_state_get(struct a...
3,3,0,net,1,int hw_atl_utils_hw_get_regs(struct aq_hw_s *s...
4,4,0,net,1,"int hw_atl_utils_initfw(struct aq_hw_s *self, ..."
5,5,0,net,1,static u32 aq_fw1x_rpc_get(struct aq_hw_s *sel...
6,6,0,net,1,u32 hw_atl_utils_get_fw_version(struct aq_hw_s...
7,7,0,net,1,int hw_atl_write_fwcfg_dwords(struct aq_hw_s *...
8,8,0,net,1,int hw_atl_utils_mpi_get_link_status(struct aq...
9,9,0,net,1,static int hw_atl_utils_soft_reset_flb(struct ...


In [25]:
project_fold_sizes = (
    manifest_df
    .groupby(["fold", "project"], as_index=False)
    .size()
    .rename(columns={"size": "functions"})
    .sort_values(["fold", "functions"], ascending=[True, False])
)

for fold_id in range(5):
    print(f"\n{'=' * 90}")
    print(f"Fold {fold_id}: five largest held-out projects")
    display(
        project_fold_sizes.loc[
            project_fold_sizes["fold"] == fold_id
        ].head(5)
    )


Fold 0: five largest held-out projects


,fold,project,functions
27,0,linux,55509
50,0,tensorflow,4805
25,0,libxml2,1933
10,0,cimg,1646
53,0,znc,798



Fold 1: five largest held-out projects


,fold,project,functions
230,1,vim,5051
170,1,mongo,4113
192,1,php-src,3121
179,1,net,2479
193,1,postgres,2292



Fold 2: five largest held-out projects


,fold,project,functions
386,2,qemu,6421
294,2,gpac,4689
245,2,FFmpeg,2821
391,2,samba,2263
287,2,ghostpdl,2192



Fold 3: five largest held-out projects


,fold,project,functions
587,3,server,7093
564,3,openssl,5416
581,3,radare2,3006
513,3,kvm,1637
585,3,redis,1461



Fold 4: five largest held-out projects


,fold,project,functions
711,4,linux-2.6,8154
657,4,envoy,5559
723,4,mysql-server,3227
618,4,ImageMagick,3203
635,4,ceph,2560


In [26]:
project_concentration = (
    manifest_df
    .groupby(["fold", "project"], as_index=False)
    .size()
    .rename(columns={"size": "functions"})
)

fold_sizes = (
    manifest_df
    .groupby("fold", as_index=False)
    .size()
    .rename(columns={"size": "fold_functions"})
)

project_concentration = project_concentration.merge(
    fold_sizes,
    on="fold",
    how="left",
)

project_concentration["share_of_fold"] = (
    project_concentration["functions"]
    / project_concentration["fold_functions"]
)

largest_project_per_fold = (
    project_concentration
    .sort_values(
        ["fold", "functions"],
        ascending=[True, False],
    )
    .groupby("fold", as_index=False)
    .head(1)
    .sort_values("fold")
)

display(
    largest_project_per_fold[
        ["fold", "project", "functions", "fold_functions", "share_of_fold"]
    ].style.format(
        {"share_of_fold": "{:.2%}"}
    )
)

,fold,project,functions,fold_functions,share_of_fold
27,0,linux,55509,69256,80.15%
230,1,vim,5051,51771,9.76%
386,2,qemu,6421,46043,13.95%
587,3,server,7093,47064,15.07%
711,4,linux-2.6,8154,47533,17.15%


In [27]:
import case_study_1.evaluation as evaluation

print("Evaluation version:", evaluation.EVALUATION_VERSION)

Evaluation version: cs1-evaluation-v1


In [28]:
# Do a small smoke test now

# This does not evaluate EXP-0 yet. It only proves the module works before we connect it to Logistic Regression.
import numpy as np
import pandas as pd
import case_study_1.evaluation as evaluation

rng = np.random.default_rng(42)

smoke_test_frames = []

for fold_id in range(5):
    labels = np.array([0] * 90 + [1] * 10)
    rng.shuffle(labels)

    scores = np.clip(
        0.05 + 0.70 * labels + rng.normal(0, 0.15, size=len(labels)),
        0.0,
        1.0,
    )

    smoke_test_frames.append(
        pd.DataFrame(
            {
                "source_row_id": np.arange(
                    fold_id * 100,
                    (fold_id + 1) * 100,
                ),
                "fold": fold_id,
                "label": labels,
                "y_score": scores,
                "project": [f"demo_project_{fold_id}"] * 100,
            }
        )
    )

smoke_predictions = pd.concat(
    smoke_test_frames,
    ignore_index=True,
)

smoke_evaluation = evaluation.evaluate_oof_predictions(
    smoke_predictions,
    config=evaluation.EvaluationConfig(
        threshold=0.50,
        expected_n_folds=5,
    ),
)

print(
    evaluation.format_metric_report(
        smoke_evaluation["pooled_metrics"]
    )
)

display(smoke_evaluation["fold_metrics"])

Pooled Out-of-Fold Evaluation
                   n_samples: 500
                vulnerable_1: 50
            non_vulnerable_0: 450
               positive_rate: 0.100000
                   threshold: 0.500000
    average_precision_pr_auc: 1.000000
                   precision: 0.980392
                      recall: 1.000000
                          f1: 0.990099
                         mcc: 0.989047
                 specificity: 0.997778
         false_positive_rate: 0.002222
               true_negative: 449
              false_positive: 1
              false_negative: 0
               true_positive: 50


,fold,n_samples,vulnerable_1,non_vulnerable_0,positive_rate,threshold,average_precision_pr_auc,precision,recall,f1,...,negative_predictive_value,false_positive_rate,false_negative_rate,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_positive_rate,test_unique_projects
0,0,100,10,90,0.1,0.5,1.0,1.000000,1.0,1.000000,...,1.0,0.000000,0.0,90,0,0,10,10,0.10,1
1,1,100,10,90,0.1,0.5,1.0,1.000000,1.0,1.000000,...,1.0,0.000000,0.0,90,0,0,10,10,0.10,1
2,2,100,10,90,0.1,0.5,1.0,1.000000,1.0,1.000000,...,1.0,0.000000,0.0,90,0,0,10,10,0.10,1
3,3,100,10,90,0.1,0.5,1.0,1.000000,1.0,1.000000,...,1.0,0.000000,0.0,90,0,0,10,10,0.10,1
4,4,100,10,90,0.1,0.5,1.0,0.909091,1.0,0.952381,...,1.0,0.011111,0.0,89,1,0,10,11,0.11,1


In [29]:
import case_study_1.exp0_lr as exp0_lr
import case_study_1.evaluation as evaluation

print("EXP-0 version:", exp0_lr.EXP0_VERSION)

EXP-0 version: cs1-exp0-sgd-logistic-v1-profiled


In [30]:
import pandas as pd
from pathlib import Path

NORMALIZED_DATA_PATH = (
    PROCESSED_DIR / "rdiversevul_cs1_normalized_v1.parquet"
)

MANIFEST_PATH = (
    MANIFEST_DIR / "cs1_project_grouped_5fold_manifest.parquet"
)

normalized_df = pd.read_parquet(NORMALIZED_DATA_PATH)
manifest_df = pd.read_parquet(MANIFEST_PATH)

print("Normalized dataset:", normalized_df.shape)
print("Manifest:", manifest_df.shape)

print("\nDataset projects:", normalized_df["project"].nunique())
print("Manifest folds:", sorted(manifest_df["fold"].unique()))
print("Dataset vulnerable rate:", f"{normalized_df['label'].mean():.4%}")

Normalized dataset: (261667, 5)
Manifest: (261667, 4)

Dataset projects: 797
Manifest folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Dataset vulnerable rate: 5.3266%


In [31]:
import case_study_1.exp0_lr as exp0_lr
import case_study_1.evaluation as evaluation

exp0_config = exp0_lr.Exp0Config(
    experiment_name="cs1_exp0_lr_sgd",

    # Same official grouped-CV protocol.
    n_splits=5,
    random_state=42,
    decision_threshold=0.50,

    # Keep exactly the same reduced, feasible TF-IDF representation.
    word_ngram_range=(1, 3),
    word_min_df=3,
    word_max_df=0.995,
    word_max_features=50_000,

    char_analyzer="char",
    char_ngram_range=(3, 4),
    char_min_df=8,
    char_max_df=0.995,
    char_max_features=60_000,

    # Logistic Regression trained through stochastic gradient descent.
    sgd_loss="log_loss",
    sgd_penalty="l2",
    sgd_alpha=1e-5,
    sgd_class_weight="balanced",
    sgd_max_iter=80,
    sgd_tol=1e-3,
    sgd_average=True,

    top_features_per_direction=30,
    verbose=True,
)

print(exp0_config)

Exp0Config(experiment_name='cs1_exp0_lr_sgd', code_column='normalized_code', source_id_column='source_row_id', label_column='label', project_column='project', fold_column='fold', n_splits=5, random_state=42, decision_threshold=0.5, word_ngram_range=(1, 3), word_min_df=3, word_max_df=0.995, word_max_features=50000, char_analyzer='char', char_ngram_range=(3, 4), char_min_df=8, char_max_df=0.995, char_max_features=60000, lowercase=False, sublinear_tf=True, tfidf_norm='l2', sgd_loss='log_loss', sgd_penalty='l2', sgd_alpha=1e-05, sgd_class_weight='balanced', sgd_max_iter=80, sgd_tol=0.001, sgd_average=True, top_features_per_direction=30, verbose=True)


In [32]:
profile_result_sgd = exp0_lr.run_exp0_profile_fold(
    normalized_frame=normalized_df,
    manifest=manifest_df,
    fold_id=0,
    config=exp0_config,
)

print("\nFold 0 computational profile:")
display(
    profile_result_sgd["training_metadata"][
        [
            "fold",
            "word_tfidf_seconds",
            "char_tfidf_seconds",
            "sparse_join_seconds",
            "model_fit_seconds",
            "prediction_seconds",
            "total_fold_seconds",
            "word_features",
            "char_features",
            "total_features",
            "model_n_iter",
            "convergence_warning_count",
            "optimizer",
        ]
    ]
)

print("\nFold 0 descriptive metrics — not official final results:")
print(
    evaluation.format_metric_report(
        profile_result_sgd["profile_metrics"]
    )
)

[15:48:27] CS1-EXP0 profiling mode: running Fold 1/5 only.
[15:48:33] Fold 1/5 started | train=192,411, test=69,256, train projects=743, test projects=54.
[15:48:33] Fold 1/5 | fitting word TF-IDF...
[15:52:00] Fold 1/5 | word TF-IDF done in 3.46 min (50,000 features).
[15:52:00] Fold 1/5 | fitting character TF-IDF...
[15:58:55] Fold 1/5 | character TF-IDF done in 6.91 min (60,000 features).
[15:58:55] Fold 1/5 | joining sparse feature matrices...
[15:58:58] Fold 1/5 | sparse matrices ready in 3.4s (total features: 110,000).
[15:58:59] Fold 1/5 | training SGD Logistic Regression (loss=log_loss, max_iter=80, alpha=1e-05)...
[15:59:22] Fold 1/5 | SGD Logistic Regression done in 22.6s (epochs=20, convergence warnings=0).
[15:59:22] Fold 1/5 | scoring held-out projects...
[15:59:22] Fold 1/5 | scoring done in 0.2s.
[15:59:22] Fold 1/5 completed in 10.82 min.
[15:59:23] Profiling run complete. These metrics are descriptive for one fold only; do not treat them as the official five-fold EXP-0

,fold,word_tfidf_seconds,char_tfidf_seconds,sparse_join_seconds,model_fit_seconds,prediction_seconds,total_fold_seconds,word_features,char_features,total_features,model_n_iter,convergence_warning_count,optimizer
0,0,207.380625,414.572584,3.385942,22.56555,0.181941,649.440231,50000,60000,110000,20,0,SGDClassifier(loss=log_loss)



Fold 0 descriptive metrics — not official final results:
Pooled Out-of-Fold Evaluation
                   n_samples: 69256
                vulnerable_1: 3397
            non_vulnerable_0: 65859
               positive_rate: 0.049050
                   threshold: 0.500000
    average_precision_pr_auc: 0.115622
                   precision: 0.135939
                      recall: 0.317339
                          f1: 0.190342
                         mcc: 0.144672
                 specificity: 0.895960
         false_positive_rate: 0.104040
               true_negative: 59007
              false_positive: 6852
              false_negative: 2319
               true_positive: 1078


In [34]:
from pathlib import Path
import case_study_1.exp0_lr as exp0_lr
import case_study_1.evaluation as evaluation

EXP0_OFFICIAL_OUTPUT_DIR = (
    OUTPUT_DIR / "official_sgd_logistic_v1"
)
EXP0_OFFICIAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

exp0_results = exp0_lr.run_exp0(
    normalized_frame=normalized_df,
    manifest=manifest_df,
    config=exp0_config,
    output_dir=EXP0_OFFICIAL_OUTPUT_DIR,
    additional_metadata={
        "run_type": "official_full_5fold_oof_evaluation",
        "normalized_dataset_path": str(NORMALIZED_DATA_PATH),
        "manifest_path": str(MANIFEST_PATH),
        "split_protocol": (
            "Fixed 5-fold StratifiedGroupKFold manifest, grouped by project, "
            "random_state=42"
        ),
        "classifier": (
            "SGDClassifier(loss='log_loss'): L2-regularized, "
            "class-weighted linear Logistic Regression"
        ),
        "profile_note": (
            "Fold 0 profile was used only to confirm computational feasibility "
            "and convergence before the complete run."
        ),
    },
)

print(
    evaluation.format_metric_report(
        exp0_results["evaluation"]["pooled_metrics"]
    )
)

[16:03:40] CS1-EXP0 official run started: 5-fold grouped CV.
[16:03:40] Configuration: word<= 50,000, char<= 60,000, optimizer=SGDClassifier(log_loss), max_iter=80, alpha=1e-05.
[16:03:42] Fold 1/5 started | train=192,411, test=69,256, train projects=743, test projects=54.
[16:03:42] Fold 1/5 | fitting word TF-IDF...
[16:07:06] Fold 1/5 | word TF-IDF done in 3.40 min (50,000 features).
[16:07:06] Fold 1/5 | fitting character TF-IDF...
[16:14:00] Fold 1/5 | character TF-IDF done in 6.90 min (60,000 features).
[16:14:00] Fold 1/5 | joining sparse feature matrices...
[16:14:02] Fold 1/5 | sparse matrices ready in 2.1s (total features: 110,000).
[16:14:03] Fold 1/5 | training SGD Logistic Regression (loss=log_loss, max_iter=80, alpha=1e-05)...
[16:14:27] Fold 1/5 | SGD Logistic Regression done in 24.6s (epochs=20, convergence warnings=0).
[16:14:27] Fold 1/5 | scoring held-out projects...
[16:14:28] Fold 1/5 | scoring done in 0.1s.
[16:14:28] Fold 1/5 completed in 10.77 min.
[16:14:29] Fol

In [35]:
display(exp0_results["evaluation"]["fold_metrics"])

display(
    exp0_results["fold_training"][
        [
            "fold",
            "train_rows",
            "test_rows",
            "word_features",
            "char_features",
            "total_features",
            "word_tfidf_seconds",
            "char_tfidf_seconds",
            "model_fit_seconds",
            "total_fold_seconds",
            "model_n_iter",
            "convergence_warning_count",
        ]
    ]
)

print(
    evaluation.format_metric_report(
        exp0_results["evaluation"]["pooled_metrics"]
    )
)

,fold,n_samples,vulnerable_1,non_vulnerable_0,positive_rate,threshold,average_precision_pr_auc,precision,recall,f1,...,negative_predictive_value,false_positive_rate,false_negative_rate,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_positive_rate,test_unique_projects
0,0,69256,3397,65859,0.049050,0.5,0.115622,0.135939,0.317339,0.190342,...,0.962186,0.104040,0.682661,59007,6852,2319,1078,7930,0.114503,54
1,1,51771,2798,48973,0.054046,0.5,0.136373,0.118663,0.481058,0.190368,...,0.964084,0.204133,0.518942,38976,9997,1452,1346,11343,0.219099,188
2,2,46043,2325,43718,0.050496,0.5,0.122479,0.117901,0.422366,0.184344,...,0.964390,0.168054,0.577634,36371,7347,1343,982,8329,0.180896,192
3,3,47064,2667,44397,0.056668,0.5,0.165362,0.153955,0.488939,0.234174,...,0.964684,0.161407,0.511061,37231,7166,1363,1304,8470,0.179968,181
4,4,47533,2751,44782,0.057876,0.5,0.142984,0.147782,0.391130,0.214514,...,0.958387,0.138560,0.608870,38577,6205,1675,1076,7281,0.153178,182


,fold,train_rows,test_rows,word_features,char_features,total_features,word_tfidf_seconds,char_tfidf_seconds,model_fit_seconds,total_fold_seconds,model_n_iter,convergence_warning_count
0,0,192411,69256,50000,60000,110000,203.793081,414.181110,24.552248,646.009425,20,0
1,1,209896,51771,50000,60000,110000,205.999716,412.733310,20.200434,642.787412,16,0
2,2,215624,46043,50000,60000,110000,206.951207,410.122031,21.512727,642.308350,17,0
3,3,214603,47064,50000,60000,110000,213.894244,400.234244,25.349385,642.701624,19,0
4,4,214134,47533,50000,60000,110000,218.867123,404.343062,29.526233,656.022938,21,0


Pooled Out-of-Fold Evaluation
                   n_samples: 261667
                vulnerable_1: 13938
            non_vulnerable_0: 247729
               positive_rate: 0.053266
                   threshold: 0.500000
    average_precision_pr_auc: 0.134451
                   precision: 0.133463
                      recall: 0.415124
                          f1: 0.201986
                         mcc: 0.159142
                 specificity: 0.848354
         false_positive_rate: 0.151646
               true_negative: 210162
              false_positive: 37567
              false_negative: 8152
               true_positive: 5786


In [36]:
display(
    exp0_results["evaluation"]["fold_summary"].loc[
        exp0_results["evaluation"]["fold_summary"]["metric"].isin(
            [
                "average_precision_pr_auc",
                "precision",
                "recall",
                "f1",
                "mcc",
                "false_positive_rate",
            ]
        )
    ]
)

,metric,mean,std,min,max
2,average_precision_pr_auc,0.136564,0.019413,0.115622,0.165362
3,precision,0.134848,0.016452,0.117901,0.153955
4,recall,0.420166,0.070461,0.317339,0.488939
5,f1,0.202748,0.021047,0.184344,0.234174
6,mcc,0.160318,0.022002,0.144664,0.197124
11,false_positive_rate,0.155239,0.037055,0.104040,0.204133


In [37]:
threshold_table = exp0_results["evaluation"]["threshold_metrics"]

display(
    threshold_table.loc[
        threshold_table["threshold"].isin(
            [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
        ),
        [
            "threshold",
            "precision",
            "recall",
            "f1",
            "mcc",
            "predicted_positive",
            "predicted_positive_rate",
            "false_positive_rate",
        ],
    ]
)

,threshold,precision,recall,f1,mcc,predicted_positive,predicted_positive_rate,false_positive_rate
9,0.1,0.068829,0.879538,0.127667,0.101180,178108,0.680667,0.669478
19,0.2,0.085512,0.744727,0.153408,0.133572,121387,0.463899,0.448099
29,0.3,0.100044,0.619027,0.172250,0.146054,86242,0.329587,0.313302
39,0.4,0.115511,0.510834,0.188417,0.153868,61639,0.235563,0.220075
49,0.5,0.133463,0.415124,0.201986,0.159142,43353,0.165680,0.151646
59,0.6,0.151977,0.322643,0.206626,0.156957,29590,0.113083,0.101292
69,0.7,0.177178,0.236835,0.202708,0.152776,18631,0.071201,0.061882
79,0.8,0.207146,0.146004,0.171282,0.135338,9824,0.037544,0.031442
89,0.9,0.252562,0.060123,0.097126,0.100576,3318,0.012680,0.010011


In [38]:
stable_features = (
    exp0_results["top_features"]
    .groupby(
        ["direction", "feature_type", "feature"],
        as_index=False,
    )
    .agg(
        folds_present=("fold", "nunique"),
        mean_coefficient=("coefficient", "mean"),
        mean_abs_coefficient=("abs_coefficient", "mean"),
    )
    .sort_values(
        ["direction", "folds_present", "mean_abs_coefficient"],
        ascending=[True, False, False],
    )
)

display(
    stable_features.loc[
        stable_features["direction"] == "vulnerable_associated"
    ].head(30)
)

,direction,feature_type,feature,folds_present,mean_coefficient,mean_abs_coefficient
150,vulnerable_associated,word,word::state_,4,3.787069,3.787069
137,vulnerable_associated,word,word::open_flags,4,3.677722,3.677722
105,vulnerable_associated,word,word::cJSON,4,3.150996,3.150996
115,vulnerable_associated,word,word::duint32,4,3.073210,3.073210
109,vulnerable_associated,word,word::context_handle,4,3.040890,3.040890
157,vulnerable_associated,word,word::wasm_,4,3.014974,3.014974
126,vulnerable_associated,word,word::jas_malloc,4,3.004920,3.004920
102,vulnerable_associated,word,word::ax25_dev,4,2.886397,2.886397
148,vulnerable_associated,word,word::sprintf buf,4,2.883526,2.883526
118,vulnerable_associated,word,word::face,4,2.876783,2.876783
